In [ ]:
# 1. Environment
!pip install -q --upgrade ultralytics==8.4.143

import torch, ultralytics

print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 2. Configuration + dataset sanity check
from pathlib import Path
from collections import Counter
import cv2
import numpy as np

ROOT = Path("/kaggle/input/datasets/sarasova/resized-1024/resized_1024")
IMG_DIR = ROOT / "images"
MASK_DIR = ROOT / "labels"

CLASS_NAMES = [
    "background",
    "stroma",
    "adk_invasion",
    "high_grade",
    "low_grade",
    "normal_gland",
]
NC = len(CLASS_NAMES)
IMG_SIZE = 1024
SEED = 42

images = sorted(IMG_DIR.glob("*.png"))
masks = sorted(MASK_DIR.glob("*.png"))

print("Images:", len(images))
print("Masks :", len(masks))

image_names = {p.name for p in images}
mask_names = {p.name for p in masks}
print("Images without masks:", len(image_names - mask_names))
print("Masks without images:", len(mask_names - image_names))

assert len(images) == len(masks) > 0
assert image_names == mask_names

all_ids = set()
counts = Counter()

for p in masks:
    m = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if m is None:
        raise RuntimeError(f"Could not read {p}")
    if m.ndim != 2:
        raise RuntimeError(f"{p.name}: expected single-channel mask, got {m.shape}")
    vals, n = np.unique(m, return_counts=True)
    all_ids.update(vals.tolist())
    for v, c in zip(vals, n):
        counts[int(v)] += int(c)

print("\nUnique mask IDs:", sorted(all_ids))
assert all_ids.issubset(set(range(NC))), f"Unexpected mask IDs: {sorted(all_ids)}"

total = sum(counts.values())
print("\nPixel counts:")
for cid in sorted(counts):
    print(f"{cid:>3}: {counts[cid]:>12,} pixels ({100 * counts[cid] / total:.3f}%)")

def slide_id(filename):
    return filename.split(".svs_")[0] if ".svs_" in filename else Path(filename).stem

slides = sorted({slide_id(p.name) for p in images})
print("\nUnique source slides:", len(slides))


In [ ]:
# 3. Slide-level split + YOLO semantic directory creation
from collections import defaultdict
import shutil
import pandas as pd
import yaml

OUT = Path("/kaggle/working/semantic_split")

groups = defaultdict(list)
for img in images:
    groups[slide_id(img.name)].append(img.name)

slides = sorted(groups)
print("Images:", len(images))
print("Source slides:", len(slides))

slide_pixels = {}
slide_tiles = {}

for i, slide in enumerate(slides):
    c = np.zeros(NC, dtype=np.int64)
    for filename in groups[slide]:
        mask = cv2.imread(str(MASK_DIR / filename), cv2.IMREAD_UNCHANGED)
        vals, nums = np.unique(mask, return_counts=True)
        unexpected = set(vals.tolist()) - set(range(NC))
        if unexpected:
            raise RuntimeError(f"{filename}: unexpected IDs {unexpected}")
        for v, n in zip(vals, nums):
            c[int(v)] += int(n)
    slide_pixels[slide] = c
    slide_tiles[slide] = len(groups[slide])
    if (i + 1) % 10 == 0:
        print(f"Analysed {i + 1}/{len(slides)} slides")

global_counts = sum(slide_pixels.values())
global_props = global_counts / global_counts.sum()

print("\nGlobal class distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i} {name:<15} {100 * global_props[i]:6.2f}%")

rng = np.random.default_rng(SEED)
n_slides = len(slides)
n_val = 9
n_test = 9
n_train = n_slides - n_val - n_test
target_tile_fracs = np.array([0.70, 0.15, 0.15])
best_score = np.inf
best_split = None
slides_arr = np.array(slides)

def split_stats(split_slides):
    counts_ = np.zeros(NC, dtype=np.int64)
    tiles_ = 0
    for s in split_slides:
        counts_ += slide_pixels[s]
        tiles_ += slide_tiles[s]
    props_ = counts_ / counts_.sum()
    return counts_, props_, tiles_

for _ in range(10000):
    perm = rng.permutation(slides_arr)
    train_slides = perm[:n_train]
    val_slides = perm[n_train:n_train + n_val]
    test_slides = perm[n_train + n_val:]

    tr_count, tr_prop, tr_tiles = split_stats(train_slides)
    va_count, va_prop, va_tiles = split_stats(val_slides)
    te_count, te_prop, te_tiles = split_stats(test_slides)

    if np.any(tr_count == 0) or np.any(va_count == 0) or np.any(te_count == 0):
        continue

    class_score = (
        np.abs(tr_prop - global_props).mean()
        + np.abs(va_prop - global_props).mean()
        + np.abs(te_prop - global_props).mean()
    )

    tile_fracs = np.array([tr_tiles, va_tiles, te_tiles], dtype=float)
    tile_fracs /= tile_fracs.sum()
    size_score = np.abs(tile_fracs - target_tile_fracs).mean()

    score = class_score + 0.5 * size_score

    if score < best_score:
        best_score = score
        best_split = {
            "train": sorted(train_slides.tolist()),
            "val": sorted(val_slides.tolist()),
            "test": sorted(test_slides.tolist()),
        }

assert best_split is not None

print("\nChosen split")
print("=" * 70)
for split, split_slides in best_split.items():
    counts_, props_, n_tiles = split_stats(split_slides)
    print(f"\n{split.upper()}: {len(split_slides)} slides, {n_tiles} tiles")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {i} {name:<15} {100 * props_[i]:6.2f}%")

assert not (set(best_split["train"]) & set(best_split["val"]))
assert not (set(best_split["train"]) & set(best_split["test"]))
assert not (set(best_split["val"]) & set(best_split["test"]))
print("\nNO SOURCE-SLIDE LEAKAGE ✓")

if OUT.exists():
    shutil.rmtree(OUT)

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

manifest = []
for split, split_slides in best_split.items():
    for slide in split_slides:
        for filename in groups[slide]:
            shutil.copy2(IMG_DIR / filename, OUT / "images" / split / filename)
            shutil.copy2(MASK_DIR / filename, OUT / "labels" / split / filename)
            manifest.append({"filename": filename, "source_slide": slide, "split": split})

manifest = pd.DataFrame(manifest)
manifest_path = Path("/kaggle/working/semantic_split_manifest.csv")
manifest.to_csv(manifest_path, index=False)

data_yaml = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "masks_dir": "labels",
    "channels": 3,
    "nc": NC,
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

yaml_path = Path("/kaggle/working/semantic_data.yaml")
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\nDataset ready:", OUT)
print("\nYAML:\n", yaml_path.read_text())
print("Manifest:", manifest_path)


In [ ]:
# 4. Train YOLO26 medium semantic model at 1024
from ultralytics import YOLO

DATA_YAML = "/kaggle/working/semantic_data.yaml"
PROJECT = "/kaggle/working/yolo_semantic_runs"
RUN_NAME = "yolo26m_sem_1024"

model = YOLO("yolo26m-sem.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=1024,
    batch=2,
    device=0,
    workers=4,
    seed=SEED,
    deterministic=True,
    cls_pw=0.5,
    mosaic=0.0,
    hsv_s=0.20,
    hsv_v=0.20,
    degrees=180,
    flipud=0.5,
    fliplr=0.5,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience=25,
    save=True,
    plots=True,
    cache=False,
    verbose=True,
)

print("\nTRAINING FINISHED")


In [ ]:
# 5. Recover best.pt and copy it to an obvious path
from pathlib import Path
import shutil

BEST = Path("/kaggle/working/yolo_semantic_runs/yolo26m_sem_1024/weights/best.pt")
FINAL = Path("/kaggle/working/yolo26m_semantic_1024_best.pt")

assert BEST.exists(), f"Could not find {BEST}"
shutil.copy2(BEST, FINAL)

print("Original best:", BEST)
print("Easy-to-find copy:", FINAL)
print(f"Size: {FINAL.stat().st_size / 1024**2:.1f} MB")


In [ ]:
# 6. Final held-out test
from ultralytics import YOLO

best_model = YOLO("/kaggle/working/yolo26m_semantic_1024_best.pt")

metrics = best_model.val(
    data="/kaggle/working/semantic_data.yaml",
    split="test",
    imgsz=1024,
    batch=1,
    device=0,
    project="/kaggle/working/yolo_semantic_runs",
    name="final_test",
    plots=True,
)

print("\nFINAL HELD-OUT TEST")
print("mIoU:", metrics.miou)
print("Pixel accuracy:", metrics.pixel_accuracy)


In [ ]:
# 7. Visual inference demo: overlay + legend + raw predicted mask
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image as IPImage

# BGR palette for OpenCV. Background is intentionally not overlaid.
PALETTE = [
    (0, 0, 0),          # 0 background
    (255, 255, 0),      # 1 stroma -> cyan
    (0, 0, 255),        # 2 adk_invasion -> red
    (0, 255, 0),        # 3 high_grade -> green
    (255, 0, 0),        # 4 low_grade -> blue
    (255, 0, 255),      # 5 normal_gland -> magenta
]

def colorize(mask):
    lut = np.zeros((256, 3), np.uint8)
    for i, col in enumerate(PALETTE):
        lut[i] = col
    return lut[mask.clip(0, 255)]

def draw_legend(img, class_ids, names, scale=1.0):
    class_ids = [c for c in class_ids if c != 0]
    if not class_ids:
        return img

    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = 0.6 * scale
    ft = max(1, round(scale))
    pad = int(8 * scale)
    sw = int(18 * scale)
    gap = int(8 * scale)

    labels = [(c, names[c]) for c in class_ids]
    sizes = [cv2.getTextSize(n, font, fs, ft)[0] for _, n in labels]
    row_h = max(sw, max(h for _, h in sizes)) + gap
    box_w = pad * 2 + sw + gap + max(w for w, _ in sizes)
    box_h = pad * 2 + row_h * len(labels) - gap
    x0, y0 = int(10 * scale), int(10 * scale)

    overlay = img.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.6, img, 0.4, 0, img)
    cv2.rectangle(img, (x0, y0), (x0 + box_w, y0 + box_h), (255, 255, 255), ft)

    y = y0 + pad
    for (cid, name), _ in zip(labels, sizes):
        cv2.rectangle(img, (x0 + pad, y), (x0 + pad + sw, y + sw), PALETTE[cid], -1)
        cv2.putText(
            img, name,
            (x0 + pad + sw + gap, y + sw - int(3 * scale)),
            font, fs, (255, 255, 255), ft, cv2.LINE_AA
        )
        y += row_h
    return img

# Default demo: one untouched held-out test tile.
# Replace this with any new same-centre PNG path later.
TEST_IMAGE = sorted(Path("/kaggle/working/semantic_split/images/test").glob("*.png"))[0]
WEIGHTS = "/kaggle/working/yolo26m_semantic_1024_best.pt"
ALPHA = 0.50

demo_model = YOLO(WEIGHTS)
r = demo_model.predict(source=str(TEST_IMAGE), imgsz=1024, device=0, verbose=False)[0]

mask = r.semantic_mask
mask = getattr(mask, "data", mask)
if hasattr(mask, "cpu"):
    mask = mask.cpu().numpy()
mask = np.asarray(mask).astype(np.int32)

img = r.orig_img.copy()
color = colorize(mask)
fg = mask != 0

blended = img.copy()
blended[fg] = (ALPHA * color[fg] + (1 - ALPHA) * img[fg]).astype(np.uint8)

present = sorted(int(c) for c in np.unique(mask))
draw_legend(blended, present, r.names, scale=max(1.0, max(img.shape[:2]) / 1200.0))

OUT_DIR = Path("/kaggle/working/demo_prediction")
OUT_DIR.mkdir(exist_ok=True)

overlay_path = OUT_DIR / f"{TEST_IMAGE.stem}_semantic.png"
mask_path = OUT_DIR / f"{TEST_IMAGE.stem}_mask.png"

cv2.imwrite(str(overlay_path), blended)
cv2.imwrite(str(mask_path), mask.astype(np.uint8))

print("Input:", TEST_IMAGE)
print("Overlay:", overlay_path)
print("Raw predicted mask:", mask_path)

total_pixels = mask.size
print("\nPredicted pixel coverage:")
for c in present:
    pct = 100 * (mask == c).sum() / total_pixels
    print(f"{c} {r.names[c]:<16} {pct:6.2f}%")

display(IPImage(filename=str(overlay_path)))


## Using a new same-centre PNG

In the final cell, replace:

```python
TEST_IMAGE = sorted(Path("/kaggle/working/semantic_split/images/test").glob("*.png"))[0]
```

with the path to any new same-centre PNG, e.g.:

```python
TEST_IMAGE = Path("/kaggle/input/your-new-dataset/new_case.png")
```

The cell saves:
- `*_semantic.png` — H&E + coloured semantic overlay + legend
- `*_mask.png` — raw predicted class-ID mask
